# 01. 학습 최적화 템플릿 — RTX 4070 SUPER (언록) 12GB

이 노트북 하나로 **학습 속도·메모리 최적화 기법을 직접 켜고 끄며 측정**합니다. 위에서부터 순서대로 실행하세요.

| # | 기법 | 섹션 |
|---|---|---|
| 10 | TF32 matmul | 1 |
| 11 · 37 | BF16 autocast · FP16 + GradScaler | 2 |
| 12 | FlashAttention 2 / SDPA | 3 |
| 13 | FP8 (Transformer Engine) | 4 |
| 15 · 16 · 17 · 36 | fused/8-bit Adam · 그래디언트 체크포인팅 · torch.compile · 그래디언트 누적 | 5 |
| 31 · 38 | DataLoader 튜닝 · DALI · CUDA 스트림 프리패치 | 6 |
| 32 | PYTORCH_CUDA_ALLOC_CONF | 7 |
| 19 | datasets 디스크 캐싱 | 8 |
| 30 | Nsight Systems / Compute | 9 |
| 33 | jupyterlab-nvdashboard | 왼쪽 사이드바 **GPU Dashboards** 탭 |

> 💡 실행하는 동안 왼쪽 **GPU Dashboards → GPU Utilization / GPU Memory** 를 열어 두면 각 기법의 효과가 그래프로 바로 보입니다.
> ⚠️ LLM 서빙(`17_serve.sh`)이 켜져 있으면 VRAM 이 부족합니다. 먼저 `sudo bash scripts/17_serve.sh stop` 을 실행하세요.

In [ ]:
import os, gc, time, math, json
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.checkpoint import checkpoint

dev = "cuda"
print(f"torch {torch.__version__} | CUDA {torch.version.cuda} | {torch.cuda.get_device_name()}")
free, total = torch.cuda.mem_get_info()
print(f"VRAM 여유 {free/2**30:.1f} / {total/2**30:.1f} GB")
print("자동 주입 상태 → TF32:", torch.backends.cuda.matmul.allow_tf32,
      "| cudnn.benchmark:", torch.backends.cudnn.benchmark,
      "| ALLOC_CONF:", os.environ.get("PYTORCH_CUDA_ALLOC_CONF"))
if free < 8 * 2**30:
    print("⚠️ 여유 VRAM 이 8GB 미만입니다 — 서빙 컨테이너나 다른 커널을 종료하세요.")

def bench(fn, iters=20, warmup=5):
    for _ in range(warmup):
        fn()
    torch.cuda.synchronize()
    t = time.perf_counter()
    for _ in range(iters):
        fn()
    torch.cuda.synchronize()
    return (time.perf_counter() - t) / iters

def cleanup():
    gc.collect(); torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()

## 1. TF32 matmul [10]
FP32 텐서를 그대로 두고 matmul 내부만 TF32 텐서코어로 계산합니다(가수 10비트). 학습 정확도 영향은 거의 없고, **언록된 텐서코어 경로**를 타므로 FP32 대비 크게 빨라집니다.
코드 한 줄이면 됩니다(이 서버에서는 startup 스크립트가 이미 켜 둠):
```python
torch.backends.cuda.matmul.allow_tf32 = True   # 또는 torch.set_float32_matmul_precision("high")
```

In [ ]:
N = 8192
a = torch.randn(N, N, device=dev); b = torch.randn(N, N, device=dev)
res = {}
for name, tf32 in [("FP32 (TF32 끔)", False), ("TF32", True)]:
    torch.backends.cuda.matmul.allow_tf32 = tf32
    res[name] = 2 * N**3 / bench(lambda: a @ b, iters=10) / 1e12
torch.backends.cuda.matmul.allow_tf32 = True
ah, bh = a.bfloat16(), b.bfloat16()
res["BF16"] = 2 * N**3 / bench(lambda: ah @ bh, iters=10) / 1e12
for k, v in res.items():
    print(f"{k:<16} {v:7.1f} TFLOPS")
del a, b, ah, bh; cleanup()

## 2·5. 학습 루프 벤치마크 — AMP · 옵티마이저 · 체크포인팅 · compile · 누적 [11][15][16][17][36][37]
작은 GPT(8층, d=1024, ~1.2억 파라미터)를 합성 데이터로 학습하며 설정별 **ms/step · tokens/s · 최대 VRAM** 을 비교합니다.

| 설정 | 기대 효과 |
|---|---|
| `amp="bf16"` | 활성값 VRAM 절반 + 텐서코어 → 빠름. 스케일러 불필요 (권장 기본값) |
| `amp="fp16"` + GradScaler | BF16 미지원 GPU 용. 언더플로 방지를 위해 **GradScaler 필수** |
| `opt="adamw_fused"` | 옵티마이저 커널 융합 → 스텝 시간↓ |
| `opt="adam8bit"` | bitsandbytes 8-bit 상태 → 옵티마이저 VRAM 약 1/4 |
| `ckpt=True` | 활성값 저장 대신 역전파 때 재계산 → VRAM↓, 시간 ~20–30%↑ |
| `compile=True` | 커널 융합 → +10~30% (첫 스텝 컴파일 시간 소요) |
| `accum=k` | 작은 배치 k번 누적 = 실효 배치 ×k (12GB 로 큰 배치 효과) |

In [ ]:
class TinyGPT(nn.Module):
    def __init__(self, vocab=32000, d=1024, L=8, h=16, ckpt=False):
        super().__init__()
        self.ckpt = ckpt
        self.emb = nn.Embedding(vocab, d)
        self.layers = nn.ModuleList([
            nn.TransformerEncoderLayer(d, h, 4 * d, dropout=0.0, batch_first=True, norm_first=True)
            for _ in range(L)])
        self.norm = nn.LayerNorm(d)
        self.head = nn.Linear(d, vocab, bias=False)

    def forward(self, x):
        h = self.emb(x)
        mask = nn.Transformer.generate_square_subsequent_mask(x.size(1), device=x.device)
        for layer in self.layers:
            f = lambda t, layer=layer: layer(t, src_mask=mask, is_causal=True)
            h = checkpoint(f, h, use_reentrant=False) if (self.ckpt and self.training) else f(h)
        return self.head(self.norm(h))

def make_opt(name, params, lr=3e-4):
    if name == "adamw":
        return torch.optim.AdamW(params, lr=lr, foreach=False)
    if name == "adamw_fused":
        return torch.optim.AdamW(params, lr=lr, fused=True)
    if name == "adam8bit":
        import bitsandbytes as bnb
        return bnb.optim.AdamW8bit(params, lr=lr)
    raise ValueError(name)

def train_bench(amp=None, opt="adamw_fused", ckpt=False, compile=False, batch=8, accum=1, seq=512, steps=8):
    cleanup(); torch.manual_seed(0)
    model = TinyGPT(ckpt=ckpt).to(dev).train()
    o = make_opt(opt, model.parameters())
    fwd = torch.compile(model) if compile else model
    dtype = {"bf16": torch.bfloat16, "fp16": torch.float16}.get(amp)
    scaler = torch.cuda.amp.GradScaler(enabled=(amp == "fp16"))       # [37]
    x = torch.randint(0, 32000, (batch, seq), device=dev)
    y = torch.randint(0, 32000, (batch, seq), device=dev)

    def step():
        for _ in range(accum):                                       # [36]
            with torch.autocast("cuda", dtype=dtype, enabled=dtype is not None):   # [11]
                loss = F.cross_entropy(fwd(x).flatten(0, 1).float(), y.flatten()) / accum
            scaler.scale(loss).backward()
        scaler.unscale_(o)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(o); scaler.update(); o.zero_grad(set_to_none=True)

    try:
        for _ in range(4 if compile else 2):
            step()
        torch.cuda.synchronize(); t = time.perf_counter()
        for _ in range(steps):
            step()
        torch.cuda.synchronize(); dt = (time.perf_counter() - t) / steps
        r = dict(ms=dt * 1000, tok_s=batch * seq * accum / dt, peak_gb=torch.cuda.max_memory_allocated() / 2**30)
    except torch.cuda.OutOfMemoryError:
        r = dict(ms=float("nan"), tok_s=0.0, peak_gb=float("nan"), oom=True)
    del model, o, fwd; cleanup()
    return r

results = {}
def run(label, **kw):
    r = train_bench(**kw); results[label] = r
    tag = "  ❌ OOM" if r.get("oom") else ""
    print(f"{label:<34} {r['ms']:8.1f} ms/step {r['tok_s']:9.0f} tok/s  peak {r['peak_gb']:5.2f} GB{tag}")

In [ ]:
# AMP 비교 [11][37]  (TF32 는 켜진 상태)
run("FP32(TF32) + AdamW", amp=None, opt="adamw")
run("BF16 autocast + AdamW", amp="bf16", opt="adamw")
run("FP16 autocast + GradScaler", amp="fp16", opt="adamw")

In [ ]:
# 옵티마이저 [15] · 체크포인팅 [16] · compile [17]
run("BF16 + fused AdamW", amp="bf16", opt="adamw_fused")
run("BF16 + 8-bit AdamW", amp="bf16", opt="adam8bit")
run("BF16 + fused + grad ckpt", amp="bf16", opt="adamw_fused", ckpt=True)
run("BF16 + fused + torch.compile", amp="bf16", opt="adamw_fused", compile=True)

In [ ]:
# 그래디언트 누적 [36] — 실효 배치 32 를 만드는 세 가지 방법
run("batch32 at once (may OOM)", amp="bf16", batch=32)
run("batch8 x accum4", amp="bf16", batch=8, accum=4)
run("batch8 x accum4 + grad ckpt", amp="bf16", batch=8, accum=4, ckpt=True)

In [ ]:
import matplotlib.pyplot as plt
ok = {k: v for k, v in results.items() if not v.get("oom")}
fig, ax = plt.subplots(1, 2, figsize=(13, max(3, 0.45 * len(ok))))
names = list(ok)
ax[0].barh(names, [ok[n]["tok_s"] for n in names], color="#4C78A8"); ax[0].set_xlabel("tokens/s (higher is better)")
ax[1].barh(names, [ok[n]["peak_gb"] for n in names], color="#F58518"); ax[1].set_xlabel("peak VRAM GB (lower is better)")
ax[1].set_yticklabels([]); ax[0].invert_yaxis(); ax[1].invert_yaxis()
plt.tight_layout(); plt.show()
json.dump(results, open("train_bench_results.json", "w"), ensure_ascii=False, indent=1)

## 3. FlashAttention 2 / SDPA [12]
NGC 컨테이너에는 FlashAttention 2 가 **이미 내장**되어 있습니다. `F.scaled_dot_product_attention` 은 조건(BF16/FP16, head_dim ≤ 256 등)이 맞으면 자동으로 Flash 커널을 고릅니다.
HF transformers 에서는 `attn_implementation="flash_attention_2"` (또는 `"sdpa"`) 로 사용합니다.

In [ ]:
from torch.nn.attention import sdpa_kernel, SDPBackend
B, H, S, D = 4, 16, 4096, 64
q, k, v = (torch.randn(B, H, S, D, device=dev, dtype=torch.bfloat16) for _ in range(3))
flops = 4 * B * H * S * S * D / 2      # causal
for name, be in [("MATH (기본 구현)", SDPBackend.MATH), ("EFFICIENT (xFormers 계열)", SDPBackend.EFFICIENT_ATTENTION),
                 ("FLASH (FlashAttention 2)", SDPBackend.FLASH_ATTENTION)]:
    try:
        with sdpa_kernel(be):
            t = bench(lambda: F.scaled_dot_product_attention(q, k, v, is_causal=True), iters=10, warmup=3)
        print(f"{name:<28} {t*1000:7.2f} ms  {flops/t/1e12:6.1f} TFLOPS")
    except Exception as e:
        print(f"{name:<28} 사용 불가: {str(e)[:80]}")
try:
    import flash_attn
    from flash_attn import flash_attn_func
    qf, kf, vf = (t.transpose(1, 2).contiguous() for t in (q, k, v))
    t = bench(lambda: flash_attn_func(qf, kf, vf, causal=True), iters=10, warmup=3)
    print(f"{'flash_attn ' + flash_attn.__version__ + ' 직접 호출':<28} {t*1000:7.2f} ms  {flops/t/1e12:6.1f} TFLOPS")
except Exception as e:
    print("flash_attn 패키지:", e)
del q, k, v; cleanup()

## 4. FP8 — Transformer Engine [13]
Ada(sm_89) 텐서코어는 FP8 을 지원합니다. NGC 에 내장된 Transformer Engine 의 `te.Linear` + `fp8_autocast` 로 GEMM 을 FP8 로 실행합니다.
대형 GEMM 에서 BF16 대비 추가 가속이 나오며, 작은 행렬은 스케일링 오버헤드 때문에 이득이 적습니다.

In [ ]:
try:
    import transformer_engine.pytorch as te
    from transformer_engine.common.recipe import DelayedScaling, Format
    recipe = DelayedScaling(fp8_format=Format.HYBRID, amax_history_len=16, amax_compute_algo="max")
    M = K = Nn = 8192
    lin = te.Linear(K, Nn, bias=False, params_dtype=torch.bfloat16).to(dev)
    x = torch.randn(M, K, device=dev, dtype=torch.bfloat16)
    flops = 2 * M * K * Nn
    with torch.no_grad():
        t_bf16 = bench(lambda: lin(x), iters=10)
        def f8():
            with te.fp8_autocast(enabled=True, fp8_recipe=recipe):
                lin(x)
        t_fp8 = bench(f8, iters=10)
    print(f"BF16 te.Linear : {flops/t_bf16/1e12:6.1f} TFLOPS")
    print(f"FP8  te.Linear : {flops/t_fp8/1e12:6.1f} TFLOPS  (×{t_bf16/t_fp8:.2f})")
    del lin, x
except Exception as e:
    print("Transformer Engine / FP8 사용 불가:", e)
cleanup()

## 6. 데이터 파이프라인 — DataLoader · CUDA 스트림 · DALI [31][38]
GPU 가 아무리 빨라도 **데이터가 늦게 오면 GPU 는 논다**(GPU Dashboards 의 사용률이 톱니 모양). 세 단계로 개선합니다.
1. `num_workers` · `pin_memory` · `persistent_workers` · `prefetch_factor` — CPU 전처리 병렬화 + 고정 메모리
2. **CUDA 스트림 프리패처** — 다음 배치의 H2D 복사를 계산과 겹침 [38]
3. **DALI** — JPEG 디코딩·리사이즈를 GPU 에서 수행 [31]

In [ ]:
from torch.utils.data import Dataset, DataLoader

class SynthImages(Dataset):
    """CPU 전처리 비용이 있는 합성 이미지 데이터셋 (uint8 → 정규화·뒤집기)"""
    def __init__(self, n=4096, size=224):
        self.n, self.size = n, size
    def __len__(self):
        return self.n
    def __getitem__(self, i):
        g = torch.Generator().manual_seed(i)
        img = torch.randint(0, 255, (3, self.size + 32, self.size + 32), dtype=torch.uint8, generator=g)
        y0, x0 = torch.randint(0, 32, (2,), generator=g).tolist()
        img = img[:, y0:y0 + self.size, x0:x0 + self.size].float().div_(255)
        if i % 2:
            img = img.flip(-1)
        img = (img - 0.45) / 0.225
        return img, i % 1000

ncpu = os.cpu_count()
def loader_time(**kw):
    dl = DataLoader(SynthImages(), batch_size=64, shuffle=True, **kw)
    it = iter(dl); next(it)                        # 워커 기동 제외
    t = time.perf_counter(); n = 0
    for x, y in it:
        x = x.to(dev, non_blocking=True); n += 1
        if n == 40: break
    torch.cuda.synchronize()
    return (time.perf_counter() - t) / n * 1000

print(f"CPU {ncpu}개")
for kw in [dict(num_workers=0),
           dict(num_workers=min(8, ncpu)),
           dict(num_workers=min(8, ncpu), pin_memory=True, persistent_workers=True, prefetch_factor=4)]:
    print(f"{str(kw):<85} {loader_time(**kw):7.1f} ms/batch")

In [ ]:
class CUDAPrefetcher:
    """다음 배치를 별도 CUDA 스트림에서 미리 GPU 로 복사 → 학습 계산과 겹침 [38]"""
    def __init__(self, loader):
        self.it = iter(loader); self.stream = torch.cuda.Stream(); self._load()
    def _load(self):
        try:
            x, y = next(self.it)
        except StopIteration:
            self.batch = None; return
        with torch.cuda.stream(self.stream):
            self.batch = (x.to(dev, non_blocking=True).to(memory_format=torch.channels_last),
                          y.to(dev, non_blocking=True))
    def __iter__(self):
        return self
    def __next__(self):
        torch.cuda.current_stream().wait_stream(self.stream)
        if self.batch is None:
            raise StopIteration
        x, y = self.batch
        x.record_stream(torch.cuda.current_stream()); y.record_stream(torch.cuda.current_stream())
        self._load()
        return x, y

import torchvision
def cnn_train(prefetch, steps=40):
    cleanup()
    m = torchvision.models.resnet50(num_classes=1000).to(dev).to(memory_format=torch.channels_last)
    o = torch.optim.SGD(m.parameters(), lr=0.1, momentum=0.9)
    dl = DataLoader(SynthImages(), batch_size=64, shuffle=True, num_workers=min(8, ncpu),
                    pin_memory=True, persistent_workers=True, prefetch_factor=4)
    src = CUDAPrefetcher(dl) if prefetch else ((x.to(dev).to(memory_format=torch.channels_last), y.to(dev)) for x, y in dl)
    n = 0
    for x, y in src:
        if n == 5:
            torch.cuda.synchronize(); t = time.perf_counter()
        with torch.autocast("cuda", dtype=torch.bfloat16):
            loss = F.cross_entropy(m(x), y)
        loss.backward(); o.step(); o.zero_grad(set_to_none=True)
        n += 1
        if n == steps + 5: break
    torch.cuda.synchronize()
    ips = steps * 64 / (time.perf_counter() - t)
    del m, o; cleanup()
    return ips

print(f"ResNet-50 BF16 · 동기 복사      : {cnn_train(False):7.0f} img/s")
print(f"ResNet-50 BF16 · 스트림 프리패치 : {cnn_train(True):7.0f} img/s")

In [ ]:
# DALI: GPU JPEG 디코딩 [31] — 데모용 JPEG 512장 생성 후 torchvision(CPU) vs DALI(GPU) 비교
from pathlib import Path
from PIL import Image
import numpy as np
root = Path("/tmp/dali_demo")
if not root.exists():
    for c in range(4):
        (root / f"class{c}").mkdir(parents=True, exist_ok=True)
        for i in range(128):
            Image.fromarray(np.random.randint(0, 255, (480, 640, 3), np.uint8)).save(root / f"class{c}/{i}.jpg", quality=90)

import torchvision.transforms as T
tv = torchvision.datasets.ImageFolder(root, T.Compose([T.RandomResizedCrop(224), T.RandomHorizontalFlip(), T.ToTensor(),
                                                         T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])]))
dl = DataLoader(tv, batch_size=64, num_workers=min(8, ncpu), pin_memory=True)
t = time.perf_counter()
for x, y in dl:
    x = x.to(dev, non_blocking=True)
torch.cuda.synchronize(); print(f"torchvision (CPU 디코딩): {len(tv)/(time.perf_counter()-t):7.0f} img/s")

try:
    from nvidia.dali import pipeline_def, fn, types
    from nvidia.dali.plugin.pytorch import DALIGenericIterator
    @pipeline_def(batch_size=64, num_threads=4, device_id=0)
    def pipe():
        jpegs, labels = fn.readers.file(file_root=str(root), random_shuffle=True, name="R")
        imgs = fn.decoders.image_random_crop(jpegs, device="mixed", output_type=types.RGB)
        imgs = fn.resize(imgs, resize_x=224, resize_y=224)
        imgs = fn.crop_mirror_normalize(imgs, dtype=types.FLOAT, output_layout="CHW", mirror=fn.random.coin_flip(),
                                        mean=[0.485 * 255, 0.456 * 255, 0.406 * 255], std=[0.229 * 255, 0.224 * 255, 0.225 * 255])
        return imgs, labels
    p = pipe(); p.build()
    it = DALIGenericIterator(p, ["x", "y"], reader_name="R")
    for _ in it: pass          # 워밍업 1 epoch
    it.reset(); t = time.perf_counter()
    for b in it:
        x = b[0]["x"]
    torch.cuda.synchronize(); print(f"DALI (GPU 디코딩)      : {len(tv)/(time.perf_counter()-t):7.0f} img/s")
    del p, it
except Exception as e:
    print("DALI 사용 불가:", e)
cleanup()

## 7. 캐싱 얼로케이터 튜닝 — `PYTORCH_CUDA_ALLOC_CONF` [32]
PyTorch 는 해제된 GPU 메모리를 바로 반환하지 않고 캐시합니다. 크기가 들쭉날쭉한 텐서(가변 길이 시퀀스 등)를 반복 할당하면 **조각화**로 "여유는 있는데 OOM" 이 납니다.

| 설정 | 효과 |
|---|---|
| `expandable_segments:True` (이 서버 기본 주입) | 세그먼트를 늘려 쓰는 방식 → 조각화 대폭 감소 |
| `max_split_size_mb:128` | 큰 블록 분할 제한 (구버전 대안) |
| `garbage_collection_threshold:0.8` | 사용률 80% 넘으면 캐시 회수 |

⚠️ 커널이 **CUDA 를 처음 쓰기 전에** 정해져야 합니다. 바꾸려면: 커널 재시작 → 첫 셀에서 `os.environ["PYTORCH_CUDA_ALLOC_CONF"]="..."` → 이후 torch 사용.

In [ ]:
cleanup()
torch.manual_seed(0)
keep = []
for i in range(300):                         # 가변 크기 할당/해제 반복 → 조각화 유도
    t = torch.empty(int(torch.randint(1, 64, (1,))) * 2**20, dtype=torch.uint8, device=dev)
    if i % 3 == 0:
        keep.append(t)
s = torch.cuda.memory_stats()
print("ALLOC_CONF        :", os.environ.get("PYTORCH_CUDA_ALLOC_CONF"))
print(f"할당(실사용)       : {s['allocated_bytes.all.current']/2**30:.2f} GB")
print(f"예약(캐시 포함)    : {s['reserved_bytes.all.current']/2**30:.2f} GB")
print(f"조각난 비활성 블록 : {s.get('inactive_split_bytes.all.current', 0)/2**20:.0f} MB  ← 작을수록 좋음")
del keep, t; cleanup()
# 자세히: print(torch.cuda.memory_summary(abbreviated=True))

## 8. datasets 디스크 캐싱 [19]
`datasets` 는 `map()` 결과를 Arrow 파일로 캐시합니다. 같은 전처리를 다시 실행하면 **계산 없이 즉시 로드**됩니다.
이 서버는 `HF_HOME` 이 영구 볼륨(`hf_cache`)이라 컨테이너를 재생성해도 캐시가 남습니다.

In [ ]:
from datasets import Dataset, load_from_disk
cache_root = os.path.join(os.environ.get("HF_HOME", os.path.expanduser("~/.cache/huggingface")), "datasets", "lab_demo")
if not os.path.exists(cache_root):
    Dataset.from_dict({"text": [f"샘플 문장 {i} " * 20 for i in range(200_000)]}).save_to_disk(cache_root)
ds = load_from_disk(cache_root)

def heavy(batch):
    return {"n_tokens": [len(t.split()) for t in batch["text"]], "upper": [t.upper() for t in batch["text"]]}

for run_i in (1, 2):
    t = time.perf_counter()
    out = ds.map(heavy, batched=True, num_proc=min(4, ncpu), load_from_cache_file=True, desc=f"map #{run_i}")
    print(f"{run_i}회차: {time.perf_counter()-t:6.2f}s  {'(캐시 적중)' if run_i == 2 else '(계산)'}")
print("캐시 파일:", out.cache_files[0]["filename"] if out.cache_files else "-")

## 9. Nsight Systems / Compute — 병목을 눈으로 [30]
최적화의 출발점은 **어디가 느린지 측정**하는 것입니다.
- **Nsight Systems (`nsys`)**: 타임라인 — CPU·CUDA API·커널·메모리 복사·NVTX 구간이 시간축에 겹쳐 보임 → 데이터 대기, 동기화, 작은 커널 난립 발견
- **Nsight Compute (`ncu`)**: 커널 하나의 심층 분석 — SM/메모리 사용률(Speed of Light), 점유율, 병목 유형
- 생성된 `.nsys-rep` / `.ncu-rep` 를 내려받아 PC 의 Nsight GUI 로 열면 상세 타임라인을 볼 수 있습니다.

In [ ]:
os.makedirs("profiles", exist_ok=True)
script = '''
import torch, torch.nn as nn, torch.nn.functional as F
m = nn.Sequential(*[nn.Linear(4096, 4096) for _ in range(8)]).cuda()
o = torch.optim.AdamW(m.parameters(), fused=True)
x = torch.randn(256, 4096, device="cuda")
for step in range(20):
    torch.cuda.nvtx.range_push(f"step{step}")
    torch.cuda.nvtx.range_push("forward")
    with torch.autocast("cuda", dtype=torch.bfloat16):
        loss = m(x).float().pow(2).mean()
    torch.cuda.nvtx.range_pop()
    torch.cuda.nvtx.range_push("backward"); loss.backward(); torch.cuda.nvtx.range_pop()
    torch.cuda.nvtx.range_push("optimizer"); o.step(); o.zero_grad(); torch.cuda.nvtx.range_pop()
    torch.cuda.nvtx.range_pop()
torch.cuda.synchronize()
'''
open("profiles/nsys_demo.py", "w").write(script)
print("프로파일 대상 스크립트 작성 완료 → 다음 셀에서 nsys 실행")

In [ ]:
!nsys profile -t cuda,nvtx,osrt --force-overwrite true -o profiles/train_demo python profiles/nsys_demo.py 2>&1 | tail -3
!nsys stats --report cuda_gpu_kern_sum --format table profiles/train_demo.nsys-rep 2>/dev/null | head -20

In [ ]:
# Nsight Compute: GEMM 커널 1개 심층 분석 (컨테이너에 SYS_ADMIN 권한 필요 — 13 스크립트가 부여)
!ncu --kernel-name regex:"gemm|sm80|sm89|cutlass" --launch-skip 10 --launch-count 1 --section SpeedOfLight python profiles/nsys_demo.py 2>&1 | grep -E "Kernel|Throughput|Duration|ERR|==PROF==" | head -20

In [ ]:
# 대안: torch.profiler (설치 불필요) — 상위 커널 표 + Chrome/Perfetto 트레이스
from torch.profiler import profile, ProfilerActivity
m = TinyGPT(L=4).to(dev)
x = torch.randint(0, 32000, (8, 512), device=dev)
with profile(activities=[ProfilerActivity.CPU, ProfilerActivity.CUDA]) as prof:
    for _ in range(3):
        with torch.autocast("cuda", dtype=torch.bfloat16):
            m(x).float().mean().backward()
    torch.cuda.synchronize()
print(prof.key_averages().table(sort_by="cuda_time_total", row_limit=12))
prof.export_chrome_trace("profiles/torch_trace.json")   # https://ui.perfetto.dev 에서 열기
del m; cleanup()

## 정리 — 이 서버에서의 권장 기본 조합
```python
torch.backends.cuda.matmul.allow_tf32 = True          # [10] (자동 주입됨)
with torch.autocast("cuda", dtype=torch.bfloat16):    # [11] 스케일러 불필요
    ...
opt = torch.optim.AdamW(params, fused=True)           # [15] VRAM 부족하면 bnb.optim.AdamW8bit
model = torch.compile(model)                          # [17] 반복 학습이면 거의 항상 이득
DataLoader(..., num_workers=8, pin_memory=True, persistent_workers=True, prefetch_factor=4)  # [31]
# VRAM 부족 → 체크포인팅 [16] → 누적 [36] → 8-bit Adam [15] 순으로 적용
```